# 🛒 Intelligent E-Commerce IR & Recommendation System Demo

Welcome to the interactive demonstration of the **E-Commerce Information Retrieval and Recommendation System**.

### System Capabilities Showcase:
1. **Text Preprocessing**: Tokenization, regex normalization, stopword filtering, and POS-aware lemmatization.
2. **Multi-Engine Retrieval**: Inverted Index TF-IDF, Okapi BM25, and Dense Semantic Transformers (`all-MiniLM-L6-v2`).
3. **Query Expansion**: WordNet synset synonyms and Pseudo-Relevance Feedback (PRF).
4. **Hybrid Ranking**: Reciprocal Rank Fusion (RRF) combining lexical and dense retrieval.
5. **Recommendation Engines**: Item-to-item Content-Based Filtering, Collaborative Filtering via Truncated SVD, and Personalization re-ranking.
6. **Review Intelligence**: Aspect sentiment analysis, polarity distributions, and key attribute extraction via TextBlob.
7. **Catalog Deduplication**: Near-duplicate detection using MinHash Locality-Sensitive Hashing (LSH).
8. **Quantitative Evaluation**: Benchmarking Precision@K, Recall@K, F1@K, MAP, MRR, and NDCG@K.

---

## 1. Setup & Imports

Import all core modules from the `src` package.

In [ ]:
import os
import sys
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Ensure project root is in path
project_root = os.path.abspath('..')
if project_root not in sys.path:
    sys.path.insert(0, project_root)

warnings.filterwarnings('ignore')

# Preprocessing
from src.preprocessing.text_preprocessor import (
    TextPreprocessor, preprocess, clean_text, tokenize, remove_stopwords, lemmatize
)

# Indexing
from src.indexing.inverted_index import InvertedIndex
from src.indexing.embedding_index import EmbeddingIndex

# Retrieval
from src.retrieval.tfidf_retrieval import TFIDFRetriever
from src.retrieval.bm25_retrieval import BM25Retriever
from src.retrieval.semantic_retrieval import SemanticRetriever
from src.retrieval.query_expansion import QueryExpander
from src.retrieval.hybrid_ranker import HybridRanker

# Recommendations
from src.recommendation.content_based import ContentBasedRecommender
from src.recommendation.collaborative import CollaborativeRecommender
from src.recommendation.personalization import PersonalizationEngine

# Analysis
from src.analysis.review_analyzer import ReviewAnalyzer
from src.analysis.duplicate_detector import DuplicateDetector

# Evaluation
from src.evaluation.metrics import (
    precision_at_k, recall_at_k, f1_at_k,
    average_precision, mean_average_precision,
    ndcg_at_k, reciprocal_rank, mean_reciprocal_rank,
    plot_comparison
)

print("✔ All system modules loaded successfully!")

## 2. Synthetic Data Creation & Exploration

To make this demo completely standalone without requiring external multi-gigabyte raw files, we instantiate a representative e-commerce catalog of 18 products spanning Audio, Computers, Smart Home, Tablets, and Wearables.

In [ ]:
sample_products = [
    {
        'product_id': '101',
        'name': 'Sony WH-1000XM5 Wireless Noise Cancelling Headphones',
        'brand': 'Sony',
        'categories': 'Audio, Headphones, Wireless',
        'description': 'Industry-leading noise cancellation with two processors and eight microphones. Exceptional sound quality with high-resolution audio support. Up to 30-hour battery life with quick charging. Ultra-comfortable lightweight design.',
        'reviews_text': 'Incredible noise cancellation on flights. Sound is rich and bass is tight. Battery lasts all week on a single charge. Very comfortable headband.',
        'avg_rating': 4.8
    },
    {
        'product_id': '102',
        'name': 'Bose QuietComfort 45 Bluetooth Wireless Headphones',
        'brand': 'Bose',
        'categories': 'Audio, Headphones, Bluetooth',
        'description': 'Iconic quiet comfort with acoustic noise cancelling technology. High-fidelity audio with adjustable EQ. Lightweight materials with plush ear cushions. 24 hours of battery life on a single charge.',
        'reviews_text': 'Best comfort for working all day. Noise cancelling is legendary. Clear microphone calls. Great Bluetooth range.',
        'avg_rating': 4.7
    },
    {
        'product_id': '103',
        'name': 'Apple AirPods Pro 2nd Gen Wireless Earbuds',
        'brand': 'Apple',
        'categories': 'Audio, Earbuds, Wireless',
        'description': 'Active Noise Cancellation reduces unwanted background noise. Adaptive Transparency lets outside sounds in while reducing loud environmental noise. Personalized Spatial Audio with dynamic head tracking. MagSafe charging case.',
        'reviews_text': 'Seamless integration with iPhone and Mac. Spatial audio is mind-blowing. Active noise cancelling is shockingly good for tiny earbuds.',
        'avg_rating': 4.9
    },
    {
        'product_id': '104',
        'name': 'JBL Flip 6 Portable Waterproof Bluetooth Speaker',
        'brand': 'JBL',
        'categories': 'Audio, Speakers, Bluetooth, Portable',
        'description': 'Eco-friendly packaging with IP67 waterproof and dustproof design. 2-way speaker system delivering loud, crystal clear, powerful sound. 12 hours of playtime. PartyBoost compatible for pairing.',
        'reviews_text': 'Took it to the beach and pool without any worries. Deep bass and punchy sound. Long battery life and rugged build.',
        'avg_rating': 4.6
    },
    {
        'product_id': '105',
        'name': 'Ultimate Ears BOOM 3 Portable Wireless Bluetooth Speaker',
        'brand': 'Ultimate Ears',
        'categories': 'Audio, Speakers, Bluetooth, Waterproof',
        'description': 'Super-portable wireless Bluetooth speaker with 360-degree immersive sound and deep thundering bass. IP67 water and dust-proof. Floats in water. 15-hour battery life.',
        'reviews_text': 'Amazing 360 sound for outdoor parties. It actually floats in the pool! Indestructible design.',
        'avg_rating': 4.5
    },
    {
        'product_id': '106',
        'name': 'Apple MacBook Air 13-inch M2 Chip 256GB SSD',
        'brand': 'Apple',
        'categories': 'Computers, Laptops, Ultrabooks',
        'description': 'Strikingly thin design with fast M2 processor. 13.6-inch Liquid Retina display with 500 nits brightness. 18 hours of all-day battery life. MagSafe charging and silent fanless operation.',
        'reviews_text': 'Blazing fast performance for coding and video editing. The battery lasts forever. Silent and super portable.',
        'avg_rating': 4.9
    },
    {
        'product_id': '107',
        'name': 'Dell XPS 13 Laptop Intel Core i7 16GB RAM',
        'brand': 'Dell',
        'categories': 'Computers, Laptops, Windows',
        'description': 'Compact premium ultrabook featuring 12th Gen Intel Core i7 processor. 13.4-inch InfinityEdge FHD display. 16GB LPDDR5 memory and 512GB PCIe SSD for demanding multitasking and programming.',
        'reviews_text': 'Sleek aluminum body and gorgeous display. Handles large spreadsheets and compilation with ease. Keyboard feels great.',
        'avg_rating': 4.4
    },
    {
        'product_id': '108',
        'name': 'Lenovo IdeaPad 3 Budget Laptop 15.6-inch FHD',
        'brand': 'Lenovo',
        'categories': 'Computers, Laptops, Budget',
        'description': 'Affordable cheap laptop powered by AMD Ryzen 5 processor. 8GB DDR4 RAM and 256GB SSD storage. 15.6-inch anti-glare display with narrow bezels. Ideal for students and office work.',
        'reviews_text': 'Great cheap budget laptop for school work and web browsing. Fast boot times. Value for money is unbeatable.',
        'avg_rating': 4.2
    },
    {
        'product_id': '109',
        'name': 'Amazon Kindle Paperwhite 16GB e-Reader with Warm Light',
        'brand': 'Amazon',
        'categories': 'Tablets, e-Readers, Books',
        'description': 'Purpose-built tablet for reading books. 6.8-inch glare-free 300 ppi display that reads like real paper even in bright sunlight. Adjustable warm light. Waterproof design with weeks of battery life.',
        'reviews_text': 'Best device ever for reading novels. Easy on the eyes at night. Battery lasts over a month. Waterproof is awesome for reading in the bath.',
        'avg_rating': 4.8
    },
    {
        'product_id': '110',
        'name': 'Apple iPad Air 10.9-inch M1 Chip 64GB Wi-Fi',
        'brand': 'Apple',
        'categories': 'Tablets, Touchscreen, Apple',
        'description': 'Powerful tablet with Apple M1 chip. 10.9-inch Liquid Retina display with True Tone. Touch ID, 12MP Ultra Wide front camera with Center Stage. Compatible with Apple Pencil and Magic Keyboard.',
        'reviews_text': 'Smooth 60fps drawing with Apple Pencil. Super responsive. Perfect balance between iPad mini and iPad Pro.',
        'avg_rating': 4.7
    },
    {
        'product_id': '111',
        'name': 'Samsung Galaxy Tab S8 11-inch Android Tablet',
        'brand': 'Samsung',
        'categories': 'Tablets, Android, Touchscreen',
        'description': 'Versatile Android tablet with included S Pen stylus. Snapdragon 8 Gen 1 processor with 120Hz display. DeX mode provides PC-like productivity experience. Fast charging 8000mAh battery.',
        'reviews_text': 'Love the included S Pen. Screen is fast and vibrant. DeX desktop mode makes it a real laptop alternative.',
        'avg_rating': 4.5
    },
    {
        'product_id': '112',
        'name': 'Amazon Echo Dot 5th Gen Smart Speaker with Alexa',
        'brand': 'Amazon',
        'categories': 'Smart Home, Smart Speakers, Voice',
        'description': 'Best sounding Echo Dot yet with clearer vocals, deeper bass and vibrant sound. Control smart home lights, locks, and thermostats with Alexa voice commands. Built-in motion sensor and thermometer.',
        'reviews_text': 'Alexa responds quickly. Sound quality is surprisingly full for such a tiny sphere. Controls all my bedroom smart lights.',
        'avg_rating': 4.6
    },
    {
        'product_id': '113',
        'name': 'Google Nest Audio Smart Speaker with Google Assistant',
        'brand': 'Google',
        'categories': 'Smart Home, Smart Speakers, Audio',
        'description': 'Helpful smart speaker designed for music. 75mm woofer and 19mm tweeter provide crystal-clear vocals and rich bass. Stream music with Google Assistant. Multi-room audio pairing support.',
        'reviews_text': 'Google Assistant understands complex questions better than Alexa. Music sounds balanced and warm.',
        'avg_rating': 4.4
    },
    {
        'product_id': '114',
        'name': 'Apple Watch Series 8 GPS 41mm Smartwatch',
        'brand': 'Apple',
        'categories': 'Wearables, Smartwatches, Fitness',
        'description': 'Advanced health smartwatch with temperature sensing, blood oxygen monitoring, and ECG app. Crash Detection and fall detection. Comprehensive workout tracking and water resistant 50m.',
        'reviews_text': 'Great heart rate tracking during workouts. Sleep stages tracking is insightful. Seamless notification alerts.',
        'avg_rating': 4.8
    },
    {
        'product_id': '115',
        'name': 'Samsung Galaxy Watch 5 Pro Titanium Smartwatch',
        'brand': 'Samsung',
        'categories': 'Wearables, Smartwatches, Fitness',
        'description': 'Rugged outdoor smartwatch with titanium case and sapphire crystal glass. Advanced BioActive sensor for body composition and heart health. GPS route navigation and multi-day battery life.',
        'reviews_text': 'Battery lasts three full days! Titanium body does not scratch. Excellent GPS tracking for trail running.',
        'avg_rating': 4.5
    },
    {
        'product_id': '116',
        'name': 'Anker Soundcore 2 Portable Bluetooth Speaker',
        'brand': 'Anker',
        'categories': 'Audio, Speakers, Bluetooth, Portable',
        'description': 'Affordable 12W stereo sound with BassUp technology. IPX7 waterproof rating protection against rain and spills. Incredible 24-hour battery playtime. Compact wireless speaker.',
        'reviews_text': 'Amazing battery life for the price. Compact and loud. Great cheap travel speaker.',
        'avg_rating': 4.5
    },
    {
        'product_id': '117',
        'name': 'Sony WH-1000XM5 Wireless Headphones (Near Duplicate Catalog Entry)',
        'brand': 'Sony',
        'categories': 'Audio, Headphones, Wireless',
        'description': 'Industry-leading noise cancellation with dual processors and eight microphones. Sound quality with high-resolution audio support. Up to 30-hour battery life with fast quick charging. Ultra-comfortable design.',
        'reviews_text': 'Noise cancelling is top notch. Sound is rich and clear. Battery lasts all week.',
        'avg_rating': 4.8
    },
    {
        'product_id': '118',
        'name': 'Kindle Paperwhite 6.8 Display Waterproof eReader',
        'brand': 'Amazon',
        'categories': 'Tablets, e-Readers, Books',
        'description': 'Dedicated e-reader tablet for reading books. 6.8-inch glare-free 300 ppi display that reads like real paper in sunlight. Warm light adjustment. Waterproof with weeks of battery.',
        'reviews_text': 'Best reading device. Glare-free screen is great in the sun. Weeks of battery.',
        'avg_rating': 4.8
    }
]

df = pd.DataFrame(sample_products)

# Construct composite 'document' string
def build_doc(row):
    return f"{row['name']} {row['brand']} {row['categories']} {row['description']}"

df['document'] = df.apply(build_doc, axis=1)

print(f"Dataset loaded: {len(df)} products")
display(df[['product_id', 'name', 'brand', 'categories', 'avg_rating']].head(8))

## 3. Text Preprocessing Demo

Explore each stage of the linguistic preprocessing pipeline: cleaning, tokenization, stopword removal, and POS-tagged WordNet lemmatization.

In [ ]:
raw_sample = "The Sony WH-1000XM5 headphones provide TOP-TIER active noise cancelling & 30-hour battery life! Check: https://sony.com/deal"

cleaned = clean_text(raw_sample)
tokens = tokenize(cleaned)
no_stops = remove_stopwords(tokens)
lemmatized = lemmatize(no_stops)

preprocessor = TextPreprocessor()
final_string = preprocessor.process_to_string(raw_sample)

pipeline_steps = pd.DataFrame([
    {"Stage": "0. Raw Text", "Output": raw_sample},
    {"Stage": "1. Cleaned (Regex & Lowercase)", "Output": cleaned},
    {"Stage": "2. Tokenized", "Output": str(tokens)},
    {"Stage": "3. Stopwords Removed", "Output": str(no_stops)},
    {"Stage": "4. POS Lemmatized", "Output": str(lemmatized)},
    {"Stage": "5. Final Normalized Document", "Output": final_string},
])

display(pipeline_steps)

## 4. Indexing: Building TF-IDF, BM25, and Semantic Indexes

Here we construct the indexing data structures over all 18 catalog documents.

In [ ]:
documents = df['document'].tolist()
product_ids = df['product_id'].astype(str).tolist()

# 1. Inverted TF-IDF Index
processed_docs = [preprocessor.process_to_string(doc) for doc in documents]
inv_index = InvertedIndex()
inv_index.build(processed_docs)
tfidf_retriever = TFIDFRetriever(inv_index, product_ids)
print(f"✔ Inverted Index constructed with {len(inv_index.vectorizer.get_feature_names_out())} vocabulary terms.")

# 2. BM25 Index
corpus_tokens = [preprocessor.process(doc) for doc in documents]
bm25_retriever = BM25Retriever(corpus_tokens, product_ids)
print(f"✔ BM25 Index constructed for {len(corpus_tokens)} documents.")

# 3. Dense Embedding Index (SentenceTransformers)
try:
    emb_index = EmbeddingIndex()
    emb_index.build(documents, batch_size=16)
    semantic_retriever = SemanticRetriever(emb_index, product_ids)
    print(f"✔ Semantic Embedding Index constructed (Shape: {emb_index.embeddings.shape}).")
except Exception as e:
    print(f"⚠ Semantic Retriever initialization skipped ({e}).")
    emb_index = None
    semantic_retriever = None

## 5. Multi-Engine Retrieval Comparison

Let us issue a natural language query and contrast the rankings and scores returned by **TF-IDF**, **Okapi BM25**, and **Dense Semantic Search**.

In [ ]:
test_query = "wireless bluetooth headphones noise cancelling"
top_k = 5

tfidf_res = tfidf_retriever.retrieve(test_query, top_k=top_k)
bm25_res = bm25_retriever.retrieve(test_query, top_k=top_k)

id_to_name = dict(zip(df['product_id'].astype(str), df['name']))

comparison_dict = {
    'Rank': list(range(1, top_k + 1)),
    'TF-IDF Product': [id_to_name.get(pid, pid)[:40] for pid, _ in tfidf_res],
    'TF-IDF Score': [f"{s:.4f}" for _, s in tfidf_res],
    'BM25 Product': [id_to_name.get(pid, pid)[:40] for pid, _ in bm25_res],
    'BM25 Score': [f"{s:.4f}" for _, s in bm25_res],
}

if semantic_retriever:
    sem_res = semantic_retriever.retrieve(test_query, top_k=top_k)
    comparison_dict['Semantic Product'] = [id_to_name.get(pid, pid)[:40] for pid, _ in sem_res]
    comparison_dict['Semantic Score'] = [f"{s:.4f}" for _, s in sem_res]

comp_df = pd.DataFrame(comparison_dict)
print(f"Query: \"{test_query}\"")
display(comp_df)

## 6. Query Expansion: WordNet & Pseudo-Relevance Feedback (PRF)

Query expansion overcomes vocabulary mismatch by enriching the input query with synonyms and high-relevance terms.

In [ ]:
expander = QueryExpander(bm25_retriever=bm25_retriever)

query_sample = "cheap laptop"
expanded_wordnet = expander.expand(query_sample, method='wordnet')
expanded_prf = expander.expand(query_sample, method='prf')

expansion_summary = pd.DataFrame([
    {"Method": "Original Query", "Query String": query_sample},
    {"Method": "WordNet Expansion", "Query String": expanded_wordnet},
    {"Method": "PRF Expansion (BM25 feedback)", "Query String": expanded_prf},
])

display(expansion_summary)

# Compare BM25 retrieval before and after expansion
print("\n--- BM25 Results for Original Query ---")
for rank, (pid, score) in enumerate(bm25_retriever.retrieve(query_sample, top_k=3), 1):
    print(f"  {rank}. [{pid}] {id_to_name[pid]} (Score: {score:.3f})")

print("\n--- BM25 Results for WordNet Expanded Query ---")
for rank, (pid, score) in enumerate(bm25_retriever.retrieve(expanded_wordnet, top_k=3), 1):
    print(f"  {rank}. [{pid}] {id_to_name[pid]} (Score: {score:.3f})")

## 7. Hybrid Ranking: Reciprocal Rank Fusion (RRF)

Reciprocal Rank Fusion aggregates ordinal rankings across multiple retrieval methods without suffering from uncalibrated score distributions:
$$\text{RRF}(d) = \sum_{m \in M} \frac{1}{k + r_m(d)}$$

In [ ]:
retrievers = {
    'tfidf': tfidf_retriever,
    'bm25': bm25_retriever
}
if semantic_retriever:
    retrievers['semantic'] = semantic_retriever

hybrid_ranker = HybridRanker(retrievers=retrievers, rrf_k=60)
query = "portable waterproof speaker"

hybrid_details = hybrid_ranker.retrieve_with_details(query, top_k=5)

print(f"Hybrid Multi-Retriever Ranking for: \"{query}\"\n")
rrf_rows = []
for rank, (pid, rrf_score) in enumerate(hybrid_details['fused'], 1):
    rrf_rows.append({
        'Final Rank': rank,
        'Product ID': pid,
        'Product Name': id_to_name.get(pid, pid),
        'RRF Score': f"{rrf_score:.5f}"
    })

display(pd.DataFrame(rrf_rows))

## 8. Recommendation Engines & Personalization

We evaluate three complementary recommendation paradigms:
1. **Content-Based Filtering**: Item-to-item feature similarity.
2. **Collaborative Filtering**: Truncated SVD matrix factorization on user ratings.
3. **Personalization Engine**: Re-ranking based on dynamic user affinity profiles.

In [ ]:
# ── 1. Content-Based Recommendation ──────────────────────────────────────
cb_recommender = ContentBasedRecommender()
cb_recommender.build_profile(df)

target_pid = '101' # Sony Headphones
cb_recs = cb_recommender.recommend(target_pid, top_n=4)

print(f"Content-Based Recommendations similar to [{target_pid}] '{id_to_name[target_pid]}':")
for rank, (rec_id, sim) in enumerate(cb_recs, 1):
    print(f"  {rank}. [{rec_id}] {id_to_name[rec_id]} (Similarity: {sim:.4f})")

# ── 2. Collaborative Filtering (Truncated SVD) ─────────────────────────
ratings_data = [
    {'user_id': 'u1', 'product_id': '101', 'rating': 5.0},
    {'user_id': 'u1', 'product_id': '102', 'rating': 4.5},
    {'user_id': 'u1', 'product_id': '104', 'rating': 2.0},
    {'user_id': 'u2', 'product_id': '104', 'rating': 5.0},
    {'user_id': 'u2', 'product_id': '105', 'rating': 4.8},
    {'user_id': 'u2', 'product_id': '116', 'rating': 4.7},
    {'user_id': 'u3', 'product_id': '106', 'rating': 5.0},
    {'user_id': 'u3', 'product_id': '107', 'rating': 4.6},
    {'user_id': 'u3', 'product_id': '108', 'rating': 3.0},
    {'user_id': 'u4', 'product_id': '109', 'rating': 5.0},
    {'user_id': 'u4', 'product_id': '118', 'rating': 5.0},
    {'user_id': 'u5', 'product_id': '101', 'rating': 4.8},
    {'user_id': 'u5', 'product_id': '106', 'rating': 4.9},
]
ratings_df = pd.DataFrame(ratings_data)

collab = CollaborativeRecommender(n_components=2)
collab.build(ratings_df)
collab_recs = collab.recommend(user_id='u1', top_n=3)

print(f"\nCollaborative Filtering (SVD) Recommendations for User 'u1':")
for rank, (rec_id, pred_score) in enumerate(collab_recs, 1):
    print(f"  {rank}. [{rec_id}] {id_to_name.get(rec_id, rec_id)} (Predicted Score: {pred_score:.4f})")

# ── 3. Personalization Engine ─────────────────────────────────────────
pers_engine = PersonalizationEngine()
# User has interacted heavily with Apple products
pers_engine.update_history('user_apple', '106', action='view', metadata={'brand': 'Apple', 'category': 'Computers'})
pers_engine.update_history('user_apple', '110', action='rate', metadata={'brand': 'Apple', 'category': 'Tablets', 'rating': 5.0})

print("\nUser Profile Affinities:", pers_engine.get_user_profile('user_apple'))

# Personalize baseline search results
raw_results = [('107', 0.8), ('106', 0.75), ('108', 0.6)] # Dell (0.8) initially ranked above Apple MacBook (0.75)
re_ranked = pers_engine.personalize('user_apple', raw_results, df)

print("\nBefore Personalization:", [(id_to_name[pid][:20], s) for pid, s in raw_results])
print("After Personalization: ", [(id_to_name[pid][:20], round(s, 3)) for pid, s in re_ranked])

## 9. Customer Review Analysis: Sentiment & Attribute Extraction

The `ReviewAnalyzer` analyzes feedback text to derive sentiment polarity, subjectivity, and prominent noun phrases.

In [ ]:
analyzer = ReviewAnalyzer()

sample_reviews = [
    "Incredible active noise cancellation on loud flights!",
    "Sound is rich and bass is punchy and tight.",
    "Battery life lasts all week on a single quick charge.",
    "The ear cushions are soft and comfortable for hours.",
    "Slightly expensive, but premium build quality makes up for it."
]

analysis = analyzer.full_analysis("Sony WH-1000XM5", sample_reviews)

print(f"Product: {analysis['product_name']}")
print(f"Average Polarity: {analysis['sentiment']['avg_polarity']:.3f}")
print(f"Sentiment Distribution: {analysis['sentiment']['distribution']}")
print(f"Top Extracted Attributes (Noun Phrases): {analysis['attributes'][:6]}")
print("\nExtractive Review Summary:")
for s in analysis['summary'][:2]:
    print(f"  • {s}")

## 10. Catalog Deduplication: MinHash Locality-Sensitive Hashing (LSH)

E-commerce catalogs often contain duplicate or re-listed entries. `DuplicateDetector` applies MinHash shingling with LSH to detect near-duplicates in sub-quadratic time.

In [ ]:
detector = DuplicateDetector(similarity_threshold=0.80)

clusters = detector.find_duplicates_lsh(documents, product_ids)

print(f"Scanned {len(documents)} catalog entries.")
print(f"Identified {len(clusters)} near-duplicate cluster(s):\n")
for i, cluster in enumerate(clusters, 1):
    print(f"Cluster #{i} Product IDs: {cluster}")
    for pid in cluster:
        print(f"   → [{pid}] {id_to_name[pid]}")

## 11. Evaluation Suite & Performance Metrics

We evaluate retrieval quality across standard Information Retrieval metrics:
- **Precision@K**
- **Recall@K**
- **F1@K**
- **MAP (Mean Average Precision)**
- **MRR (Mean Reciprocal Rank)**
- **NDCG@K (Normalized Discounted Cumulative Gain)**

In [ ]:
# Benchmark query set with relevance ground truth
eval_queries = [
    {
        'query': 'noise cancelling wireless headphones',
        'relevant_ids': {'101', '102', '103', '117'}
    },
    {
        'query': 'portable waterproof bluetooth speaker',
        'relevant_ids': {'104', '105', '116'}
    },
    {
        'query': 'portable laptop computer for work',
        'relevant_ids': {'106', '107', '108'}
    },
    {
        'query': 'ereader tablet for reading books',
        'relevant_ids': {'109', '118'}
    },
    {
        'query': 'smartwatch fitness health tracking',
        'relevant_ids': {'114', '115'}
    }
]

k = 3
models = {
    'TF-IDF': tfidf_retriever,
    'BM25': bm25_retriever,
    'Hybrid (RRF)': hybrid_ranker
}
if semantic_retriever:
    models['Semantic'] = semantic_retriever

results_dict = {}

for model_name, retriever in models.items():
    all_retrieved = []
    all_relevant = []
    
    for eq in eval_queries:
        res = retriever.retrieve(eq['query'], top_k=k)
        retrieved_ids = [pid for pid, _ in res]
        all_retrieved.append(retrieved_ids)
        all_relevant.append(eq['relevant_ids'])
        
    p = np.mean([precision_at_k(r, rel, k) for r, rel in zip(all_retrieved, all_relevant)])
    r = np.mean([recall_at_k(r, rel, k) for r, rel in zip(all_retrieved, all_relevant)])
    f1 = np.mean([f1_at_k(r, rel, k) for r, rel in zip(all_retrieved, all_relevant)])
    map_score = mean_average_precision(all_retrieved, all_relevant)
    mrr_score = mean_reciprocal_rank(all_retrieved, all_relevant)
    ndcg_score = np.mean([ndcg_at_k(r, rel, k) for r, rel in zip(all_retrieved, all_relevant)])
    
    results_dict[model_name] = {
        f'Precision@{k}': p,
        f'Recall@{k}': r,
        f'F1@{k}': f1,
        'MAP': map_score,
        'MRR': mrr_score,
        f'NDCG@{k}': ndcg_score
    }

eval_df = pd.DataFrame(results_dict).T
print(f"Retrieval Performance Evaluation (K={k}):")
display(eval_df.round(4))

# Visualization
plt.figure(figsize=(10, 5))
eval_df[['Precision@3', 'Recall@3', 'MAP', 'NDCG@3']].plot(
    kind='bar', figsize=(11, 5), colormap='viridis', edgecolor='black'
)
plt.title(f'Comparative IR Performance Across Retrieval Methods (K={k})', fontsize=14, fontweight='bold')
plt.ylabel('Score (0.0 to 1.0)', fontsize=12)
plt.xlabel('Retrieval Model', fontsize=12)
plt.ylim(0, 1.15)
plt.xticks(rotation=0, fontsize=11)
plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left', frameon=True)
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

## 12. Summary & Conclusions

### Key Takeaways:
1. **Lexical vs. Dense Trade-offs**: TF-IDF and BM25 excel at exact keyword matches and specific product model numbers. Dense Semantic Retrieval handles synonymy and broad natural language inquiries.
2. **Hybrid Synergy**: Reciprocal Rank Fusion (RRF) leverages the distinct strengths of both paradigms, achieving robust rankings across varied query modalities.
3. **Query Expansion**: WordNet synset mapping and PRF bridge the vocabulary gap, boosting recall for sparse or short search phrases.
4. **Recommenders & Personalization**: Content-based filtering solves item cold-start, collaborative SVD uncovers latent patterns, and dynamic personalization aligns ranking with user interests.
5. **Review & Catalog Intelligence**: Near-duplicate detection preserves catalog hygiene, while aspect sentiment analysis delivers actionable product insights.